# Phase 8 — Interaction features

Phase 7's tuning ceiling sat at ROC-AUC ≈ 0.704 / KS ≈ 0.298. The notebook closed by flagging that the next gain wasn't in more hyperparameters — depth-4 trees can only catch one feature interaction per branch — but in *pre-computing* the underwriting-style combinations a credit officer would write down by hand.

**What this phase does:**
1. Adds 7 hand-designed interaction features (`features.py`) to a parallel set of parquets (`*_featv2.parquet`) — the v1 parquets stay live so `xgb_v1/v2/v3` keep loading.
2. Refits **the same hyperparameters** v3 was tuned to, so the v3 vs v4 comparison is *features-only* — same model capacity, different feature space.
3. Compares on the held-out 2017-2018 test set, decides promote / don't.

**Outputs:** `outputs/models/xgb_v4_interactions.joblib`, `data/processed/{train,test}_featv2.parquet`.

Re-running the smoke (`python src/_smoke_phase8.py`) regenerates both. The notebook below just loads the saved artifacts.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT = Path(r'C:\Users\User\Desktop\Credit Risk Prediction System')
sys.path.insert(0, str(PROJECT / 'src'))

from prepare import load_processed, load_processed_featv2
from features import INTERACTION_COLS, add_interactions
from models import split_xy, evaluate, model_path
from evaluate import gains_table

FIGS = PROJECT / 'outputs' / 'figures'
FIGS.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid')

## 1. The 7 interactions (and why each one)

Every feature in `features.py` is a *known* underwriting signal, not a random feature-cross. The point of hand-design over `PolynomialFeatures` is keeping the new features interpretable — a credit officer should be able to recognize what each one represents.

| Feature | Definition | Underwriting story |
|---|---|---|
| `loan_to_income` | `loan_amnt / annual_inc` | How big is this loan vs the borrower's income? Standard ratio. |
| `installment_to_income` | `installment·12 / annual_inc` | Approximate Debt-Service Coverage. Months of income consumed by payments. |
| `int_rate_x_term` | `int_rate · term` | Total interest exposure over loan life. Captures "expensive long loan" — neither `int_rate` nor `term` alone tells you this. |
| `fico_dti_risk` | `(850 − fico_mean) · (dti / 100)` | Monotone-risky in both arguments. Low FICO + high DTI is double-bad in a way the linear sum isn't. |
| `revol_util_x_logbal` | `(revol_util/100) · log1p(revol_bal)` | "Maxed out AND a lot of debt." Compresses the long tail on `revol_bal`. |
| `accounts_per_year` | `total_acc / max(credit_history_years, 1)` | Account-opening velocity. Aggressive new-credit-seeking. |
| `delinq_per_year` | `delinq_2yrs / max(credit_history_years, 1)` | Delinquency rate normalized by history length. Distinguishes "one delinquency in 30 years" from "two delinquencies in 1 year." |

**NaN propagation:** if any operand is NaN, the interaction is NaN. XGBoost handles this natively.

**Zero-division:** `annual_inc <= 0` happens in ~0.001% of rows. `_safe_div()` replaces these with NaN rather than `inf` so the model decides rather than fitting on an extreme value.

## 2. Load the featv2 parquets and inspect

In [ ]:
train, test = load_processed_featv2()
print(f'train: {train.shape}  default rate {train["default"].mean()*100:.2f}%')
print(f'test:  {test.shape}   default rate {test["default"].mean()*100:.2f}%')
print(f'\nNew columns: {INTERACTION_COLS}')

In [ ]:
# Sanity stats — mean, std, NaN share, linear corr with default.
stats = []
for c in INTERACTION_COLS:
    s = train[c]
    stats.append({
        'feature': c,
        'mean': s.mean(),
        'std': s.std(),
        'nan_pct': s.isna().mean() * 100,
        'corr_with_default': s.corr(train['default'].astype(float)),
    })
pd.DataFrame(stats).round(4)

**Reading the table.** `int_rate_x_term` shows a linear correlation of +0.27 with default — the strongest of the bunch. `fico_dti_risk` follows at +0.14. The two income-ratio features (`loan_to_income`, `installment_to_income`) show near-zero *linear* correlation, but they may still matter under non-linear splits — XGBoost finds them useful even when Pearson says no (see Section 5).

## 3. Distributions

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(15, 6))
for ax, col in zip(axes.flat, INTERACTION_COLS):
    s = train[col].dropna()
    lo, hi = s.quantile([0.01, 0.99])
    ax.hist(s.clip(lo, hi), bins=50, color='steelblue', alpha=0.7)
    ax.set_title(col)
axes.flat[-1].axis('off')
fig.suptitle('Interaction-feature distributions (clipped 1-99th pct)')
fig.tight_layout()
fig.savefig(FIGS / 'phase8_interactions_dist.png', dpi=120)
plt.show()

## 4. Load v3 and v4, evaluate on test

In [ ]:
v3 = joblib.load(model_path('xgb_v3_tuned'))
v4 = joblib.load(model_path('xgb_v4_interactions'))

# v3 lives on the v1 (25-feature) parquets; v4 on featv2 (32-feature).
# The test ROWS are identical, so metric comparison is valid.
_, test_v1 = load_processed()
X_test_v1, y_test_v1 = split_xy(test_v1)
X_test_v2, y_test_v2 = split_xy(test)
assert len(test_v1) == len(test), 'test row counts differ — re-run smokes'

p_v3 = v3.predict_proba(X_test_v1)[:, 1]
p_v4 = v4.predict_proba(X_test_v2)[:, 1]

m_v3 = evaluate(y_test_v1, p_v3)
m_v4 = evaluate(y_test_v2, p_v4)

pd.DataFrame({
    'xgb_v3_tuned (Phase 7)':         m_v3.as_row(),
    'xgb_v4_interactions (Phase 8)':  m_v4.as_row(),
}).T.round(4)

In [ ]:
deltas = pd.DataFrame([
    {'metric': k,
     'v3': getattr(m_v3, k),
     'v4': getattr(m_v4, k),
     'delta': getattr(m_v4, k) - getattr(m_v3, k)}
    for k in ['roc_auc', 'pr_auc', 'brier', 'log_loss', 'ks']
])
deltas['v4_better'] = deltas.apply(
    lambda r: (r['delta'] < 0) if r['metric'] in ('brier', 'log_loss') else (r['delta'] > 0),
    axis=1,
)
deltas.round(4)

Five metrics, five wins for v4 — but the deltas are small (ROC-AUC +0.0007, KS +0.0014). This is within the CV-fold noise we measured in Phase 7 (~0.01 std). Strictly better, but the gain isn't dramatic.

## 5. Did the interactions earn their keep?

Two views: feature importance by gain (which features the model splits on most often, weighted by improvement) and the number of interactions in the top 15.

In [ ]:
gain = pd.Series(v4.base.get_booster().get_score(importance_type='gain'))
top15 = gain.sort_values(ascending=False).head(15)
is_interaction = [n in INTERACTION_COLS for n in top15.index]

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['tomato' if i else 'steelblue' for i in is_interaction]
top15.iloc[::-1].plot(kind='barh', ax=ax, color=colors[::-1])
ax.set_xlabel('Gain (XGBoost)')
ax.set_title('Top 15 features in xgb_v4 — red = interaction')
fig.tight_layout()
fig.savefig(FIGS / 'phase8_importance_v4.png', dpi=120)
plt.show()

n_in_top15 = sum(is_interaction)
print(f'{n_in_top15}/7 interactions in top 15. '
      f'Top interaction: {top15[is_interaction].index[0]} (gain {top15[is_interaction].iloc[0]:.0f}).')

**`int_rate_x_term` is now the dominant feature**, beating `grade` and `sub_grade` by 3-5×. That's because `int_rate` is already LendingClub's price-of-risk signal — and the time-multiplied version (longer loan + higher rate = more total interest exposure) is a strictly more concentrated read on default risk than either alone. The trees previously had to spend two splits to capture this; pre-computing it saves depth.

The two income-ratio features (`loan_to_income`, `installment_to_income`) made the top 12 despite their near-zero *linear* correlation with default. That's the value of a non-linear classifier — Pearson misses what threshold splits catch.

## 6. Decile-1 lift comparison

Operationally the question is: among the 10% of test loans the model flags as highest risk, how many actually default? This is where the cost-curve threshold operates.

In [ ]:
gt_v3 = gains_table(y_test_v1, p_v3, n_bins=10)
gt_v4 = gains_table(y_test_v2, p_v4, n_bins=10)

compare = pd.DataFrame({
    'decile': gt_v3.index,
    'lift_v3': gt_v3['lift'].values,
    'lift_v4': gt_v4['lift'].values,
    'cum_v3': gt_v3['cum_capture_rate'].values,
    'cum_v4': gt_v4['cum_capture_rate'].values,
}).set_index('decile')
compare.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
x = np.arange(1, 11)
w = 0.35
axes[0].bar(x - w/2, compare['lift_v3'], width=w, label='v3', color='steelblue')
axes[0].bar(x + w/2, compare['lift_v4'], width=w, label='v4 (interactions)', color='tomato')
axes[0].axhline(1.0, color='grey', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Decile (1 = highest predicted risk)')
axes[0].set_ylabel('Lift over base rate')
axes[0].set_title('Lift by decile')
axes[0].legend()

axes[1].plot(x, compare['cum_v3'], marker='o', label='v3', color='steelblue')
axes[1].plot(x, compare['cum_v4'], marker='s', label='v4 (interactions)', color='tomato')
axes[1].plot([1, 10], [0.1, 1.0], '--', color='grey', alpha=0.5, label='random')
axes[1].set_xlabel('Decile')
axes[1].set_ylabel('Cumulative % defaults captured')
axes[1].set_title('Cumulative gains')
axes[1].legend()

fig.tight_layout()
fig.savefig(FIGS / 'phase8_gains_v3_vs_v4.png', dpi=120)
plt.show()

## 7. Findings + deployment status

1. **All five test metrics improved**, modestly. ROC-AUC 0.7040 → 0.7047 (+0.0007). KS 0.2975 → 0.2989 (+0.0014). PR-AUC 0.4454 → 0.4466 (+0.0012). Brier 0.1788 → 0.1786. Log-loss 0.5341 → 0.5335. Decile-1 lift 1.9261× → 1.9428×.
2. **The interactions earned their keep:** 4 of 7 are in the top 15 by gain. `int_rate_x_term` is the **#1 feature overall**, beating `grade` by 3×. The two income-ratio features made the top 12 despite near-zero linear correlation — non-linear splits found use for them.
3. **The gain is small because it had to be.** Phase 7 already showed the test-set ceiling is governed by concept drift between 2007-2016 training and 2017-2018 test. Feature engineering inside the training distribution can't close that gap.
4. **`int_rate_x_term`'s dominance is informative.** The model previously needed two splits to capture this interaction; now one split does it. That's *capacity saving*, not new information — which is why the test gain is so small. The implication: depth-4 trees were paying for this interaction multiple times.
5. **Promoted v4 to production.** Deployment defaults in `serve.py`, `score_batch.py`, `monitor.py` now point at `xgb_v4_interactions`. The API surface is unchanged — callers still submit the 25 base fields and the service computes the 7 interactions internally.

### Where the next gain actually lives

Phase 8's verdict matches Phase 7's: more features inside the training distribution won't move the test ceiling. Two genuinely different levers:

1. **Rolling recalibration in production.** Once production accumulates a year of labeled outcomes, retrain just the isotonic calibrator on rolling-12-month data. The base XGBoost stays the same; the calibrator adapts to drift. This is the cheapest fix and the one drift would respond to first.
2. **Sub-grade-conditional models.** Phase 5 showed per-grade KS varied 0.13 → 0.22 across grades. Fit small specialists per grade band (A-B, C-D, E-G) and route at score time. Each specialist sees a narrower default-rate range and can focus on the signals that separate within that band.

Neither is a tuning exercise. Both are structural model changes.

---

### Check-for-understanding

1. Why did `loan_to_income` and `installment_to_income` make the top 12 in v4 by gain despite a Pearson correlation of essentially zero with the target?
2. We held hyperparameters constant from v3 when training v4. What's the argument for *re-tuning* on the new feature set instead — and what's the counter-argument?
3. The test-set improvement was +0.0007 ROC-AUC. Phase 7 said the CV-test gap was caused by concept drift. Is the small test gain in Phase 8 *evidence for* or *evidence against* that diagnosis?
4. Suppose v4 had degraded one metric (say Brier) while improving the other four. Would you still promote it? What would the deciding question be?
5. The serve.py API keeps the 25-field schema even though the model needs 32 features. Name a concrete failure mode this design choice protects against.